# Notebook 02 — Normalization, Log-Transformation, and HVG Selection

**Project:** RetinoblastomaAtlas  
**Input:** `data/processed/02_qc_filtered.h5ad`  
**Output:** `data/processed/03_normalized.h5ad`, HVG list, PCA diagnostics

---

## Scientific Background

Raw UMI counts are dominated by **library-size effects**: a cell sequenced more deeply will have higher counts for every gene, independent of true biology. Normalization removes this technical covariate so that expression values reflect actual gene activity.

The three-step transformation applied here:

1. **Library-size normalization** (`normalize_total`): rescales each cell to a fixed total of 10,000 counts, making cells comparable regardless of sequencing depth
2. **log1p transformation**: compresses the right-skewed dynamic range of expression values, making the data closer to normally distributed (required for PCA)
3. **Scaling** (zero-mean, unit-variance): ensures highly expressed housekeeping genes don't dominate the principal components

## Why Batch-Aware HVG Selection?

Standard HVG selection on the merged matrix can be dominated by between-batch technical variability. We use Seurat v3-flavoured HVG selection with `batch_key='sample_id'`, which estimates variance **within** each sample and selects genes that are variable within samples, not just between them. This is the recommended approach before scVI integration (Lopez et al. 2018, *Nat Methods*).

**References:**  
- Hafemeister C, Satija R. Normalization and variance stabilization. *Genome Biol* 2019. https://doi.org/10.1186/s13059-019-1874-1  
- Lopez R et al. Deep generative modeling for single-cell transcriptomics. *Nat Methods* 2018. https://doi.org/10.1038/s41592-018-0229-2

## Parameters

In [1]:
N_HVG      = 3000    # Number of highly variable genes to select
N_PCS      = 50       # PCs for pre-scVI diagnostic PCA
TARGET_SUM = 10000   # Library-size normalization target (CPM-like)

## Setup

In [2]:
import warnings
from pathlib import Path

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

warnings.filterwarnings("ignore")
sc.settings.verbosity = 1

ROOT     = Path("..").resolve()
IN_H5AD  = ROOT / "data" / "processed" / "02_qc_filtered.h5ad"
OUT_H5AD = ROOT / "data" / "processed" / "03_normalized.h5ad"
FIG_DIR  = ROOT / "results" / "figures"
TAB_DIR  = ROOT / "results" / "tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

## Step 1 — Load QC-Filtered Atlas

In [3]:
print("Step 1 — Loading QC-filtered atlas")
adata = sc.read_h5ad(IN_H5AD)
print(f"  {adata.n_obs:,} cells × {adata.n_vars:,} genes")
print(f"  Layers present: {list(adata.layers.keys())}")

Step 1 — Loading QC-filtered atlas
  136,199 cells × 28,306 genes
  Layers present: ['counts']


## Step 2 — Library-Size Normalization

Each cell is scaled so that its total counts sum to 10,000 (approximately counts per million). This makes expression values directly comparable between cells with different sequencing depths without modifying the raw counts stored in `.layers['counts']`.

In [ ]:
print(f"Step 2 — Library-size normalization (target = {TARGET_SUM:,} counts/cell)")
sc.pp.normalize_total(adata, target_sum=TARGET_SUM)
print("  Normalization complete.")

Step 2 — Library-size normalization (target = 10,000 counts/cell)


## Step 3 — log1p Transformation

`log(1 + x)` is preferred over `log(x)` to handle zero counts gracefully (no undefined values). The `+1` pseudo-count also makes the transformed distribution approximately symmetric for downstream linear methods (PCA, regression). After transformation, a 10-fold change in expression corresponds to roughly 2.3 units — a much more interpretable scale than raw counts.

In [ ]:
print("Step 3 — log1p transformation")
sc.pp.log1p(adata)
adata.layers["lognorm"] = adata.X.copy()  # Store for reference
print("  log1p transformation complete. Log-normalized layer saved to .layers['lognorm']")

## Step 4 — Batch-Aware Highly Variable Gene (HVG) Selection

We select the top 3,000 HVGs using the Seurat v3 method with `batch_key='sample_id'`. This:
1. Fits a regularized negative binomial model to the mean-variance relationship **within each sample**
2. Selects genes that deviate upward from the expected variance (more variable than expected for their mean expression level)
3. Ranks genes by their **median rank** across samples, avoiding HVGs driven by a single outlier sample

For retinoblastoma, this captures: cone-specific markers (ARR3, RXRG, THRB), proliferation markers (MKI67, TOP2A), and immune markers (CD68, AIF1).

In [ ]:
print(f"Step 4 — HVG selection (Seurat v3, batch_key='sample_id', top {N_HVG:,})")
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=N_HVG,
    flavor="seurat_v3",
    layer="counts",        # use raw counts for dispersion fitting
    batch_key="sample_id", # batch-aware selection
    subset=False,          # keep all genes; HVG flag in .var
)
n_hvg = adata.var["highly_variable"].sum()
print(f"  Selected {n_hvg:,} HVGs")

# Save HVG table
hvg_df = adata.var[
    ["highly_variable", "means", "dispersions_norm", "highly_variable_rank"]
].query("highly_variable").sort_values("highly_variable_rank")
hvg_df.to_csv(TAB_DIR / "highly_variable_genes.csv")
print(f"  Saved HVG list ({len(hvg_df):,} genes)")
print(f"  Top 10 HVGs by rank: {hvg_df.head(10).index.tolist()}")

In [ ]:
# Mean-variance plot coloured by HVG status
fig, ax = plt.subplots(figsize=(8, 6))
not_hvg = ~adata.var["highly_variable"]
ax.scatter(
    adata.var.loc[not_hvg, "means"],
    adata.var.loc[not_hvg, "dispersions_norm"],
    s=2, alpha=0.3, color="#AAAAAA", label="Not HVG", rasterized=True,
)
ax.scatter(
    adata.var.loc[adata.var["highly_variable"], "means"],
    adata.var.loc[adata.var["highly_variable"], "dispersions_norm"],
    s=3, alpha=0.7, color="#E84C4C",
    label=f"HVG (n={adata.var['highly_variable'].sum():,})",
    rasterized=True,
)
ax.set_xscale("log")
ax.set_xlabel("Mean expression", fontsize=11)
ax.set_ylabel("Normalised dispersion", fontsize=11)
ax.set_title(
    f"Highly Variable Gene selection\n(Seurat v3, batch_key='sample_id', top {N_HVG:,} genes)",
    fontsize=10,
)
ax.legend(fontsize=9)
plt.tight_layout()
fig.savefig(FIG_DIR / "hvg_mean_variance.pdf", dpi=200, bbox_inches="tight")
plt.show()
print("Genes in the upper cloud (higher variance than expected) are selected as HVGs.")

## Step 5 — Scaling and Diagnostic PCA

PCA is variance-maximising: without scaling, highly expressed genes (ACTB, GAPDH, MALAT1) dominate the first principal components regardless of biological relevance. Zero-mean, unit-variance scaling (`max_value=10` caps extreme outliers) ensures all HVGs contribute equally.

**Important**: This PCA is computed on the **uncorrected** matrix as a diagnostic to visualise the batch effect before scVI integration. The actual dimensionality reduction for analysis uses the scVI latent space (notebook 03).

In [ ]:
print("Step 5 — Scaling HVG matrix and computing diagnostic PCA")
adata_hvg = adata[:, adata.var["highly_variable"]].copy()
sc.pp.scale(adata_hvg, max_value=10)
sc.tl.pca(adata_hvg, n_comps=N_PCS, svd_solver="arpack")

# Copy PCA back for reference
adata.obsm["X_pca_pre_scvi"] = adata_hvg.obsm["X_pca"]
adata.uns["pca"]              = adata_hvg.uns["pca"]
print(f"  PCA computed ({N_PCS} components on {adata_hvg.n_vars:,} HVGs)")

In [ ]:
# Scree plot
var_ratio = adata_hvg.uns["pca"]["variance_ratio"]
cum_var   = np.cumsum(var_ratio)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(range(1, len(var_ratio) + 1), var_ratio * 100, color="#4C9BE8", edgecolor="none")
axes[0].set_xlabel("Principal component", fontsize=11)
axes[0].set_ylabel("Variance explained (%)", fontsize=11)
axes[0].set_title("Scree plot", fontsize=11)

axes[1].plot(range(1, len(cum_var) + 1), cum_var * 100, color="#E84C4C", lw=2)
axes[1].axhline(80, color="grey", linestyle="--", linewidth=1)
axes[1].set_xlabel("Number of PCs", fontsize=11)
axes[1].set_ylabel("Cumulative variance (%)", fontsize=11)
axes[1].set_title("Cumulative variance explained", fontsize=11)
pc80 = int(np.searchsorted(cum_var, 0.8)) + 1
axes[1].annotate(f"80% at PC{pc80}", xy=(pc80, 80), xytext=(pc80 + 2, 78), fontsize=9,
                  arrowprops=dict(arrowstyle="->"))

plt.suptitle("PCA diagnostics (HVG-scaled matrix, pre-scVI correction)", fontsize=11, fontweight="bold")
plt.tight_layout()
fig.savefig(FIG_DIR / "pca_variance_explained.pdf", dpi=200, bbox_inches="tight")
plt.show()
print(f"  80% of variance explained by {pc80} PCs.")
print("  Note: This PCA is for diagnostics only. scVI will use all HVGs for integration.")

## Step 6 — Save Normalized Atlas

In [ ]:
print(f"Saving normalized atlas → {OUT_H5AD.name}")
adata.write_h5ad(OUT_H5AD, compression="gzip")
size_mb = OUT_H5AD.stat().st_size / 1e6
print(f"  {adata.n_obs:,} cells × {adata.n_vars:,} genes | {size_mb:.0f} MB")
print("\n  AnnData structure:")
print("    .X                      : log-normalized counts (float32)")
print("    .layers['counts']       : raw UMI counts (int) — required for scVI")
print("    .layers['lognorm']      : log-normalized copy")
print("    .var['highly_variable'] : HVG flag")
print("    .obsm['X_pca_pre_scvi'] : pre-correction PCA (diagnostic only)")
print(f"\n  Next: Run notebook 03_scvi_integration.ipynb")